In [2]:
import torch
from unsloth import FastVisionModel 
from transformers import TextStreamer
import torch
from PIL import Image
import json
import re
import os
import pandas as pd

# Load the model and processor
model, tokenizer = FastVisionModel.from_pretrained(
    "qwen_lora_REALM_desc",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

def extract_json(text):
    match = re.search(r'\{[\s\S]*\}', text)
    if not match:
        raise ValueError("No JSON found")
    return json.loads(match.group())


def analyze_image_realism(image_path):
    """
    Analyze an image for realism using Qwen3-VL-8B-Instruct
    
    Args:
        image_path: Path to the image file or PIL.Image object
    
    Returns:
        dict: JSON response with realism analysis
    """
    # Load image if path is provided
    if isinstance(image_path, str):
        image = Image.open(image_path)
    else:
        image = image_path
    
    # Your prompt for realism analysis
    prompt = (
    "Is there anything unrealistic in this image? yes or no or somewhat, "
    "if yes or somewhat explain in maximum 30 words, please ensure to explain "
    "what looks unreal like if face is distorted, or transition between objects is not smooth."
    "Respond ONLY in the following JSON format:\n"
    "{\n"
    '  "unrealistic": "yes | no | somewhat",\n'
    '  "explanation": "string"\n'
    "}\n\n"
)
    # Prepare messages for the model
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt}
            ]
        }
    ]
    
    # Process inputs
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens = False,
        return_tensors = "pt",
    )
    inputs = inputs.to(model.device)
    
    output_text = model.generate(
        **inputs,
        # streamer = TextStreamer(tokenizer, skip_prompt = True),
        max_new_tokens = 200,
        use_cache = True,
        temperature = 1.5,
        min_p = 0.1,
    )
        
    generated_ids = output_text[0][inputs["input_ids"].shape[-1]:]

    output_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    output_text = extract_json(output_text)
    
    return output_text

# Example usage
if __name__ == "__main__":

    generated_descriptions = []
    PATH_TO_IMAGE_FOLDER = r"/home/manem/Qwen-3VL-Testing/dataset/images/test_images"
    PATH_TO_OUTPUT_CSV = r"/home/manem/Qwen-3VL-Testing/descriptions/after-finetuning/test"
    images = [os.path.join(PATH_TO_IMAGE_FOLDER, img) for img in os.listdir(PATH_TO_IMAGE_FOLDER) if img.endswith(('.png'))]
    for img_path in images:
        try:
            result = analyze_image_realism(img_path)
            label = result['unrealistic']
            explanation = result['explanation']

            img_path_formatted = os.path.basename(img_path)
            generated_descriptions.append({
                'image_path': img_path_formatted,
                'unrealistic': label,
                'explanation': explanation
            })
            print(generated_descriptions[-1])
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            generated_descriptions.append({
                'image_path': os.path.basename(img_path),
                'unrealistic': "error",
                'explanation': str(e)
            })
    
    # convert generated descriptions to csv and save
    df = pd.DataFrame(generated_descriptions)
    df.to_csv(os.path.join(PATH_TO_OUTPUT_CSV, "test_descriptions_realism_unsloth.csv"), index=False)

        

==((====))==  Unsloth 2026.1.4: Fast Qwen3_Vl patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A40. Num GPUs = 1. Max memory: 44.339 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.17s/it]


{'image_path': 'f104.png', 'unrealistic': 'no', 'explanation': 'there is nothing unrealistic in this image. The architecture, water, and person all appear natural and consistent with a real-world scene.'}
{'image_path': 'f108.png', 'unrealistic': 'no', 'explanation': 'there is nothing unrealistic in this image. The cityscape, bridge, water, and boats all appear natural and proportionate.'}
{'image_path': 'f114.png', 'unrealistic': 'somewhat', 'explanation': "The eagles' heads and bodies have slightly unnatural symmetry and texture, especially the feathers on the right eagle's neck area, which appear somewhat odd and less detailed than real birds."}
{'image_path': 'f124.png', 'unrealistic': 'somewhat', 'explanation': "The bird's feet and their grip on the branch look unnatural and disproportionate, as do the edges of the feathers, which lack typical bird plumage detail."}
{'image_path': 'f126.png', 'unrealistic': 'somewhat', 'explanation': 'The leaf has an unnaturally vibrant orange-yel